## Initialise input widgets and fetch parameters

In [0]:
dbutils.widgets.text("catalog_name", "spotify_catalog")
dbutils.widgets.text("adls_storage_container_name", "")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
adls_storage_container_name = dbutils.widgets.get("adls_storage_container_name")

In [0]:
import os
import sys

project_pth = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_pth)

In [0]:
from utils.transformations import reusable_transformations

transform_obj = reusable_transformations()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## DimUser

In [0]:
df_user = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimUser/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/DimUser")

In [0]:
df_user.display()

In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name")))
df_user.display()

In [0]:
df_user = transform_obj.dropColumns(df_user, ["_rescued_data"])
df_user.display()

In [0]:
df_user = df_user.dropDuplicates(['user_id'])
df_user.display()

In [0]:
df_user.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimUser/checkpoint") \
    .trigger(once=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimUser/data") \
    .toTable(f"{catalog_name}.silver.DimUser")

## DimArtist

In [0]:
df_artist = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimArtist/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/DimArtist")

In [0]:
df_artist.display()

In [0]:
df_artist = df_artist.withColumn("artist_name", upper(col("artist_name")))
df_artist.display()

In [0]:
df_artist = transform_obj.dropColumns(df_artist, ["_rescued_data"])
df_artist = df_artist.dropDuplicates(['artist_id'])
df_artist.display()

In [0]:
df_artist.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimArtist/checkpoint") \
    .trigger(once=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimArtist/data") \
    .toTable(f"{catalog_name}.silver.DimArtist")

## DimTrack

In [0]:
df_track = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimTrack/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/DimTrack")

In [0]:
df_track.display()

In [0]:
df_track = df_track.withColumn("durationFlag", when(col("duration_sec") <= 150, "low").when((col("duration_sec") > 150) & (col("duration_sec") < 300), "medium").otherwise("high")
)

df_track.display()

In [0]:
df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"), "-", " "))
df_track.display()

In [0]:
df_track = transform_obj.dropColumns(df_track, ['_rescued_data'])
df_track = df_track.dropDuplicates(['track_id'])

In [0]:
df_track.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimTrack/checkpoint") \
    .trigger(once=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimTrack/data") \
    .toTable(f"{catalog_name}.silver.DimTrack")

## Dim_Date

In [0]:
df_date = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimDate/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/DimDate")

In [0]:
df_date.display()

In [0]:
df_date = transform_obj.dropColumns(df_date, ['_rescued_data'])

df_date.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimDate/checkpoint") \
    .trigger(once=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/DimDate/data") \
    .toTable(f"{catalog_name}.silver.DimDate")

## FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles") \
          .option("cloudFiles.format", "parquet") \
          .option("cloudFiles.schemaLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/FactStream/checkpoint") \
          .option("schemaEvolutionMode", "addNewColumns") \
          .load(f"abfss://bronze@{adls_storage_container_name}.dfs.core.windows.net/FactStream")

In [0]:
df_fact.display()

In [0]:
df_track = spark.read.table(f"{catalog_name}.silver.dimtrack")

In [0]:
df_joined_fact = df_fact.alias("df_fact").join(df_track.alias("df_track"), col("df_fact.track_id") == col("df_track.track_id"), how="left") \
          .select("df_fact.*","artist_id")

In [0]:
df_fact = transform_obj.dropColumns(df_fact, ['_rescued_data'])

In [0]:
df_fact.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/FactStream/checkpoint") \
    .trigger(once=True) \
    .option("path", f"abfss://silver@{adls_storage_container_name}.dfs.core.windows.net/FactStream/data") \
    .toTable(f"{catalog_name}.silver.FactStream")